# 第 10 周 · 周一：案例——"DAU突然跌了15%，怎么查？"

> 难度：⭐⭐⭐⭐ | 定位：掌握异动诊断的系统方法论，能在1小时内给出初步结论
> 前置依赖：第1周SQL、第3周AB实验、第6周用户增长、第8周指标体系
> 面试地位：大厂数据分析面试的压轴题，100%考察异动归因分析能力
> 配对课程：01python_learn W10D1（PCA降维）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 异动诊断的本质

异动诊断是数据分析师日常工作中最高频、最紧急的任务类型之一。它的本质是：**在一个复杂系统中，快速定位导致宏观指标变化的关键因子**。

这个问题的难点不在于技术复杂度，而在于三层约束：
- **时间约束**：通常要求在1-2小时内给出初步结论，而不是花3天做出完整分析
- **信息约束**：接到任务时往往只有一句话"DAU跌了15%"，没有更多上下文
- **置信度约束**：不需要100%确定，但需要在有限时间内给出"最可能的方向"

### 1.2 DPS拆解法（PDS Decomposition）

DPS是异动诊断的第一板斧，也是最核心的方法论。DPS代表：
- **P（Population）= 人群拆解**：把用户按行为模式分成互斥的群体，看哪个群体变了
- **D（Dimension）= 维度下钻**：沿关键维度（平台、渠道、版本、地域）向下拆分，定位到最细粒度
- **S（Signal）= 信号验证**：基于拆解结果形成假设，通过多源数据交叉验证确定根因

DAU的人群拆解公式（P层）：
```
DAU = 新增用户(N) + 留存用户(R) + 回流用户(C)
```

这三个群体有完全不同的业务含义：
- **新增用户**变了 → 渠道投放、应用商店排名、竞品动态
- **留存用户**变了 → 产品质量、服务稳定性、推送策略
- **回流用户**变了 → 运营活动、召回策略、季节性因素

### 1.3 从第一性原理推导：为什么PDS拆解是第0步？

很多人遇到DAU下降，第一反应是"拉一堆维度数据出来看"。这是效率最低的做法——因为你在100个维度中随机探索，运气好5分钟找到，运气不好2小时还没头绪。

PDS拆解把"找问题"变成了"分类问题"：
1. 先确定是新用户/老用户/回流用户哪一类出了问题（将搜索空间从100降到33）
2. 在出问题的那一类中，再沿维度下钻（将搜索空间从33降到5-8）
3. 对top可疑维度做假设验证（将搜索空间从8降到1-2）

这种"剪枝式"分析策略，本质上是决策树的思想——每一层分裂都在最大化信息增益。

### 1.4 异动诊断的四大原则

1. **先确认数据，再分析原因**：数据延迟、口径变更、埋点故障这三个"假异动"至少占了异动报警的30%
2. **先总后分，逐层下钻**：先看整体人口结构，再按维度拆，不要从最细粒度开始
3. **先排除技术问题，再看业务变化**：服务端、CDN、第三方依赖先排除，再看用户行为
4. **先给方向，再求精确**：1小时给出80%置信度的方向 > 3天给出95%置信度的精确结论

### 1.5 常见DAU异动模式

| 异动模式 | 典型表现 | 最常见原因 | 诊断时间 |
|---------|---------|-----------|--------|
| 骤降型 | 某天突然下降>10% | 技术故障、外部事件 | 30分钟内 |
| 缓降型 | 连续多天缓慢下降 | 渠道质量衰退、竞品侵蚀 | 2-4小时 |
| 周期性异动 | 每周/每月规律性波动 | 业务节奏（周末、月初） | 10分钟（识别模式） |
| 结构性异动 | 特定人群/平台异常 | 版本升级、策略变更 | 1-2小时 |

### 1.6 面试高频考点

**面试官常问**：
- "DAU跌了20%，你怎么排查？"
- "如何区分技术故障和业务变化导致的DAU下降？"
- "给你1小时，你会怎么分配时间？"
- "如果所有维度拆完后都没发现问题，下一步做什么？"

**回答要点**：
- 先确认数据（延迟/口径/埋点），再做PDS拆解，最后维度下钻
- 技术故障通常影响多个平台/渠道同时发生；业务变化通常只影响特定维度
- 时间分配：数据确认5分钟→PDS拆解10分钟→维度下钻20分钟→假设验证15分钟→写结论10分钟
- 如果所有维度都没问题，回到P层——检查是否人口基数本身在变小（整体流量池缩小）

## 🧩 区块2：知识点详细拆解

### 2.1 数据确认——异动诊断的第0步

**为什么重要**：至少30%的"异动"实际上是数据问题而非业务问题。跳过这一步就开始分析，相当于在沙子地基上建房子。

**确认清单（5分钟）**：

| 检查项 | 具体操作 | 异常标志 |
|--------|---------|--------|
| 数据延迟 | 查昨天的数据是否完全入库 | 今天的数值明显低于趋势线 |
| 口径一致 | 确认DAU用user_id还是device_id | 不同报表的DAU数值不一致 |
| 埋点正常 | 查客户端日志上报量 | 上报量突然下降但服务端正常 |
| 时区/日期 | 确认统计周期的起止时间 | 跨时区产品的隔日统计偏差 |
| ETL任务 | 查数仓任务是否全部成功 | 下游表出现空分区 |

**典型案例**：某天DAU"下降"了20%，最后发现是凌晨的ETL任务失败了，ods层的导入任务重试到中午才完成，T+1数据"迟到"了。实际DAU没降。

### 2.2 PDS拆解的数学原理

**为什么重要**：PDS拆解不只是"看一下这三个数"，而是要对变化量做归因分解。

DAU变化量的精确归因：

```
ΔDAU = ΔNew + ΔRetained + ΔResurrected
```

其中每项的贡献度 = 该项的变化量 / 总变化量。

**归因示例**：
```
前天DAU = 120万（New=12万, Retained=88万, Resurrected=20万）
昨天DAU = 102万（New=10万, Retained=72万, Resurrected=20万）

ΔDAU = -18万
ΔNew = -2万 → 贡献11%
ΔRetained = -16万 → 贡献89% ← 问题在这！
ΔResurrected = 0万 → 贡献0%
```

### 2.3 维度下钻的策略与优先级

**为什么重要**：维度下钻不是"把所有维度都拆一遍"，而是按优先级有序下钻，每一步都缩小搜索范围。

**维度下钻优先级**（按信息量排序）：

| 优先级 | 维度 | 为什么 | 能排除什么 |
|--------|------|--------|-----------|
| P0 | 平台（iOS/Android/Web） | 技术问题通常单平台，业务问题多平台 | 服务端vs客户端问题 |
| P1 | 版本（App版本号） | 新版本引入bug是最常见原因之一 | 发版故障 |
| P2 | 渠道（获客来源） | 渠道投放变化直接影响新增 | 投放问题 |
| P3 | 地域（省市/国家） | 区域性网络故障、政策变化 | 基础设施问题 |
| P4 | 用户分层（新老/活跃度） | 不同用户群受影响程度不同 | 功能/策略影响面 |

**下钻原则**：
- 先拆"能一刀切的维度"（平台、版本），再拆"连续维度"（时长、频次）
- 每次下钻后看"贡献度"——哪个子维度贡献了最多的变化量
- 如果某个子维度贡献了80%以上的变化量，直接锁定这个子维度继续拆

### 2.4 辛普森悖论在异动分析中的陷阱

**为什么重要**：辛普森悖论是异动分析中最容易踩的陷阱——总量下降，但每个子维度都在涨。

**经典场景**：
```
整体DAU下降5%

但按平台拆开看：
- iOS DAU涨了3%
- Android DAU涨了2%
- Web DAU涨了1%

怎么可能？→ 因为用户结构变了！
高活跃用户占比从70%降到了40%，低活跃用户占比从30%涨到了60%。
虽然每个平台的DAU都在涨，但"用户质量"在下降，高活跃用户的流失没有在平台维度上体现。
```

**应对方法**：维度下钻时始终保留"结构性视角"——不仅要看每个维度的变化，还要看维度之间的结构比例变化。

### 2.5 贡献度分析（Contribution Analysis）

**为什么重要**：贡献度分析是量化"哪一因素最值得关注"的核心工具。

**贡献度公式**：
```
某因素的贡献度 = 该因素导致的变化量 / 总变化量
```

但更准确的做法是双因素贡献度——同时考虑"结构性变化"和"比率性变化"：

```
ΔDAU = Σ(Δ比例_i × 基数_i + 比例_i × Δ基数_i + Δ比例_i × Δ基数_i)

第一项 = 人口结构变化贡献
第二项 = 各项自身活跃率变化贡献
第三项 = 交叉项（通常较小，可忽略或归平分）
```

### 2.6 假设验证的ISSUE框架

**为什么重要**：维度下钻告诉你"哪里变了"，但不告诉你"为什么变"。假设验证是连接"现象"和"根因"的桥梁。

**ISSUE框架**（每步10-15分钟）：

| 步骤 | 含义 | 操作 |
|------|------|------|
| I = Identify | 列出可能假设 | 基于维度下钻结果，列出3-5个最可能的假设 |
| S = Sort | 按优先级排序 | 按先验概率×影响面排序 |
| S = Single out | 逐一验证 | 对每个假设找"证伪"数据，而不是"证实"数据 |
| U = Update | 更新信念 | 根据验证结果更新对每个假设的置信度 |
| E = End | 输出结论 | 给出"最可能假设+置信度+下一步验证建议" |

**关键认知**：假设验证阶段要找的是"证伪"数据，而不是"证实"数据。如果你找的是证实数据，你总能找到支持任何假设的数据——这叫确认偏误。

### 2.7 海森堡效应——你在分析的过程中也在影响数据

**为什么重要**：这是一个很容易被忽略但非常重要的陷阱。

在异动诊断过程中，你可能会：
- 向运营确认数据口径 → 运营去后台操作 → 触发了缓存刷新 → 数据开始恢复
- 向后端询问登录服务 → 后端发现token失效bug → 紧急修复 → DAU回升

所以你永远无法知道"如果不干预会怎样"——这意味着你的归因永远有一部分是不可知的。

**应对**：在分析报告中诚实地说"在排查过程中发现了X问题并已修复"，而不是声称"问题的根因完全在于X"。

### 2.8 日常异动 vs 突发事件 vs 系统性衰退

**为什么重要**：三种异动类型需要完全不同的应对策略，区分它们决定了你的应急等级和资源配置。

| 类型 | 特征 | 信号 | 应对策略 | 响应时间 |
|------|------|------|---------|--------|
| 日常异动 | 波动<5%，恢复快 | 无其他伴随指标异常 | 记录观察，设置阈值报警 | 不需要立即行动 |
| 突发事件 | 骤降>10%，波动大 | 伴随其他指标异常（crash率、加载失败率） | 立即排查，通知oncall | 30分钟内 |
| 系统性衰退 | 缓慢持续下降 | 周趋势线持续向下 | 启动专项分析 | 1-2周内 |

**判断方法**：画过去30天的DAU置信带（均值±2标准差），如果当天DAU跌破下限，属于突发事件；如果在带内但持续走低，属于系统性衰退。

## 💻 区块3：动手练习

### 步骤0：环境与数据准备

模拟一个真实产品的DAU数据（含用户明细），用于完整的异动诊断演练。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 模拟近30天的DAU数据
n_days = 30
dates = pd.date_range(end='2024-06-15', periods=n_days, freq='D')

# 用户构成
n_total_users = 50000

# 生成用户基础属性
users = pd.DataFrame({
    'user_id': range(1, n_total_users + 1),
    'platform': np.random.choice(['iOS', 'Android', 'Web'], n_total_users, p=[0.35, 0.45, 0.20]),
    'channel': np.random.choice(['信息流', '搜索', '社媒', '自然', '邮件'], n_total_users, p=[0.30, 0.20, 0.15, 0.25, 0.10]),
    'city_tier': np.random.choice(['一线', '新一线', '二线', '三线及以下'], n_total_users, p=[0.25, 0.25, 0.25, 0.25]),
    'app_version': np.random.choice(['v3.2.0', 'v3.2.1', 'v3.3.0', 'v3.3.1'], n_total_users, p=[0.10, 0.15, 0.40, 0.35]),
    'register_date': pd.to_datetime(np.random.choice(pd.date_range('2023-01-01', '2024-06-01', freq='D'), n_total_users)),
    'user_level': np.random.choice(['高活', '中活', '低活', '沉默'], n_total_users, p=[0.15, 0.25, 0.35, 0.25])
})

print("用户基础数据生成完成")
print(f"总用户数: {n_total_users}")
print(f"\n平台分布:\n{users['platform'].value_counts()}")
print(f"\n渠道分布:\n{users['channel'].value_counts()}")
print(f"\n版本分布:\n{users['app_version'].value_counts()}")
print(f"\n城市等级分布:\n{users['city_tier'].value_counts()}")

### 步骤1：基础练习——PDS拆解实战

1.1 生成带异常的DAU日度数据
1.2 执行PDS拆解定位问题人群
1.3 输出归因分解报告

In [ ]:
# 步骤1.1：生成带异常的DAU日度数据

def generate_daily_dau_with_anomaly(users_df, dates, anomaly_day=-2, anomaly_type='retention_drop'):
    """
    生成日度DAU及PDS分解数据
    anomaly_type可选: 'retention_drop', 'new_user_drop', 'all_drop', 'none'
    """
    daily_stats = []
    
    for i, date in enumerate(dates):
        # 基础概率
        base_new_prob = 0.02
        base_retain_prob = 0.15
        base_resurrect_prob = 0.03
        
        # 异常日的特殊处理
        if i == len(dates) + anomaly_day and anomaly_type == 'retention_drop':
            base_retain_prob = 0.07  # 留存暴跌
        elif i == len(dates) + anomaly_day and anomaly_type == 'new_user_drop':
            base_new_prob = 0.005  # 新增暴跌
        elif i == len(dates) + anomaly_day and anomaly_type == 'all_drop':
            base_new_prob = 0.01
            base_retain_prob = 0.08
        
        # 星期效应
        weekday = date.weekday()
        weekend_factor = 1.15 if weekday >= 5 else 1.0
        
        # 模拟DAU
        n_new = int(n_total_users * base_new_prob * weekend_factor * np.random.uniform(0.9, 1.1))
        n_retain = int(n_total_users * base_retain_prob * weekend_factor * np.random.uniform(0.9, 1.1))
        n_resurrect = int(n_total_users * base_resurrect_prob * weekend_factor * np.random.uniform(0.9, 1.1))
        dau = n_new + n_retain + n_resurrect
        
        daily_stats.append({
            'date': date,
            'dau': dau,
            'new_users': n_new,
            'retained_users': n_retain,
            'resurrected_users': n_resurrect,
            'weekday': weekday,
            'is_weekend': weekday >= 5
        })
    
    return pd.DataFrame(daily_stats)

# 生成正常数据和异常数据(留存下降)
df_normal = generate_daily_dau_with_anomaly(users, dates, anomaly_type='none')
df_anomaly = generate_daily_dau_with_anomaly(users, dates, anomaly_type='retention_drop')

print("正常场景: 最近7天DAU")
print(df_normal[['date','dau','new_users','retained_users','resurrected_users']].tail(7).to_string(index=False))

print("\n异常场景: 最近7天DAU (留存暴跌)")
print(df_anomaly[['date','dau','new_users','retained_users','resurrected_users']].tail(7).to_string(index=False))

In [ ]:
# 步骤1.2：执行PDS拆解

def pds_decomposition(df):
    """
    PDS三人群拆解 + 贡献度分析
    返回：各人群的变化量和贡献度
    """
    # 取最后两天做对比
    yesterday = df.iloc[-1]
    day_before = df.iloc[-2]
    
    dau_change = yesterday['dau'] - day_before['dau']
    new_change = yesterday['new_users'] - day_before['new_users']
    retain_change = yesterday['retained_users'] - day_before['retained_users']
    resur_change = yesterday['resurrected_users'] - day_before['resurrected_users']
    
    print("=" * 60)
    print("PDS拆解报告")
    print("=" * 60)
    print(f"对比日期: {day_before['date'].strftime('%Y-%m-%d')} → {yesterday['date'].strftime('%Y-%m-%d')}")
    print(f"\nDAU变化: {day_before['dau']:,} → {yesterday['dau']:,} (Δ={dau_change:+,})")
    print(f"变化率: {dau_change/day_before['dau']*100:+.1f}%")
    print(f"\n--- 人群拆解 ---")
    print(f"{'人群':<12}{'前日':>10}{'昨日':>10}{'变化量':>10}{'贡献度':>10}")
    print(f"{'-'*52}")
    
    # 新增用户
    contrib_new = new_change / dau_change * 100 if dau_change != 0 else 0
    print(f"{'新增用户':<12}{day_before['new_users']:>10,}{yesterday['new_users']:>10,}{new_change:>+10,}{contrib_new:>9.1f}%")
    
    # 留存用户
    contrib_retain = retain_change / dau_change * 100 if dau_change != 0 else 0
    print(f"{'留存用户':<12}{day_before['retained_users']:>10,}{yesterday['retained_users']:>10,}{retain_change:>+10,}{contrib_retain:>9.1f}%")
    
    # 回流用户
    contrib_resur = resur_change / dau_change * 100 if dau_change != 0 else 0
    print(f"{'回流用户':<12}{day_before['resurrected_users']:>10,}{yesterday['resurrected_users']:>10,}{resur_change:>+10,}{contrib_resur:>9.1f}%")
    
    # 诊断结论
    print(f"\n--- 诊断结论 ---")
    contributions = {
        '新增用户': abs(contrib_new),
        '留存用户': abs(contrib_retain),
        '回流用户': abs(contrib_resur)
    }
    main_cause = max(contributions, key=contributions.get)
    print(f"主要问题人群: {main_cause}")
    
    return {
        'dau_change': dau_change,
        'dau_change_pct': dau_change / day_before['dau'] * 100,
        'new_change': new_change,
        'retain_change': retain_change,
        'resurrected_change': resur_change,
        'main_cause': main_cause
    }

# 对异常数据执行PDS拆解
result = pds_decomposition(df_anomaly)

In [ ]:
# 步骤1.3：可视化PDS趋势（最近14天）

def plot_pds_trend(df, title="DAU PDS拆解趋势"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    tail_df = df.tail(14)
    
    # 图1: DAU整体趋势
    ax1 = axes[0, 0]
    ax1.plot(range(14), tail_df['dau'].values, 'b-o', linewidth=2, markersize=6)
    ax1.axvline(x=12, color='r', linestyle='--', alpha=0.5, label='异常日')
    ax1.set_title('DAU趋势（近14天）', fontsize=14)
    ax1.set_xlabel('天数（相对）')
    ax1.set_ylabel('DAU')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 图2: 三人群堆叠面积图
    ax2 = axes[0, 1]
    ax2.stackplot(range(14),
                  tail_df['new_users'].values,
                  tail_df['retained_users'].values,
                  tail_df['resurrected_users'].values,
                  labels=['新增', '留存', '回流'],
                  colors=['#2ecc71', '#3498db', '#f39c12'],
                  alpha=0.8)
    ax2.axvline(x=12, color='r', linestyle='--', alpha=0.5)
    ax2.set_title('DAU人口结构（近14天）', fontsize=14)
    ax2.set_xlabel('天数（相对）')
    ax2.set_ylabel('用户数')
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    # 图3: 各人群占比变化
    ax3 = axes[1, 0]
    total = tail_df['dau'].values
    ax3.plot(range(14), tail_df['retained_users'].values / total * 100, 'b-o', label='留存占比')
    ax3.plot(range(14), tail_df['new_users'].values / total * 100, 'g-o', label='新增占比')
    ax3.plot(range(14), tail_df['resurrected_users'].values / total * 100, 'orange', marker='o', label='回流占比')
    ax3.axvline(x=12, color='r', linestyle='--', alpha=0.5)
    ax3.set_title('各人群占比变化（近14天）', fontsize=14)
    ax3.set_xlabel('天数（相对）')
    ax3.set_ylabel('占比(%)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 图4: 贡献度柱状图
    ax4 = axes[1, 1]
    result = pds_decomposition(df.tail(2))
    categories = ['新增用户', '留存用户', '回流用户']
    contributions = [
        abs(result.get('new_change', 0) or 0),
        abs(result.get('retain_change', 0) or 0),
        abs(result.get('resurrected_change', 0) or 0)
    ]
    colors = ['#2ecc71', '#e74c3c' if abs(result.get('retain_change', 0) or 0) == max(contributions) else '#3498db', '#f39c12']
    ax4.bar(categories, contributions, color=colors, alpha=0.8)
    ax4.set_title('异常日变化量贡献度', fontsize=14)
    ax4.set_ylabel('绝对变化量')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_pds_trend(df_anomaly, title="DAU异动分析——留存用户暴跌场景")

### 步骤2：进阶练习——维度下钻与贡献度追踪

2.1 多维度交叉下钻
2.2 贡献度追踪矩阵
2.3 自动化异动检测

In [ ]:
# 步骤2.1：多维度交叉下钻

def dimension_drill_down(users_df, anomaly_date, dimension='platform'):
    """
    在指定维度上进行下钻分析
    模拟异常日和非异常日在该维度上的分布变化
    """
    dim_values = users_df[dimension].unique()
    
    results = []
    for val in dim_values:
        subset = users_df[users_df[dimension] == val]
        n_in_dim = len(subset)
        
        # 正常日活跃率
        normal_active_rate = np.random.uniform(0.15, 0.25)
        # 异常日活跃率（部分维度受影响）
        if dimension == 'platform' and val in ['Android', 'Web']:
            anomaly_active_rate = normal_active_rate * np.random.uniform(0.3, 0.5)  # 暴跌
        elif dimension == 'app_version' and val == 'v3.3.1':
            anomaly_active_rate = normal_active_rate * np.random.uniform(0.2, 0.4)  # 新版本出问题
        else:
            anomaly_active_rate = normal_active_rate * np.random.uniform(0.9, 1.1)
        
        normal_users = int(n_in_dim * normal_active_rate)
        anomaly_users = int(n_in_dim * anomaly_active_rate)
        change = anomaly_users - normal_users
        change_pct = (anomaly_active_rate - normal_active_rate) / normal_active_rate * 100
        
        results.append({
            'dimension': dimension,
            'value': val,
            'normal_users': normal_users,
            'anomaly_users': anomaly_users,
            'change': change,
            'change_pct': change_pct
        })
    
    return pd.DataFrame(results)

# 三个维度的下钻
print("="*60)
print("维度下钻分析")
print("="*60)

for dim in ['platform', 'app_version', 'channel']:
    drill_result = dimension_drill_down(users, '2024-06-14', dimension=dim)
    print(f"\n--- 按{dim}下钻 ---")
    print(drill_result.to_string(index=False))
    
    # 找出贡献最大的子维度
    max_change_idx = drill_result['change'].abs().idxmax()
    worst = drill_result.iloc[max_change_idx]
    print(f"→ 最可疑: {dim}={worst['value']} (变化{worst['change']:+,}, {worst['change_pct']:+.1f}%)")

In [ ]:
# 步骤2.2：贡献度追踪矩阵——按平台×版本交叉下钻

def cross_dimension_drill(users_df):
    """
    二维交叉下钻：平台 × 版本
    找出受影响最严重的用户子群
    """
    platforms = users_df['platform'].unique()
    versions = users_df['app_version'].unique()
    
    matrix = []
    for plat in platforms:
        row = []
        for ver in versions:
            subset = users_df[(users_df['platform'] == plat) & (users_df['app_version'] == ver)]
            n_users = len(subset)
            
            # 模拟异常影响：Android+v3.3.1受影响最大
            if plat == 'Android' and ver == 'v3.3.1':
                anomaly_impact = np.random.uniform(-0.50, -0.35)  # 严重
            elif plat == 'Web' and ver == 'v3.3.1':
                anomaly_impact = np.random.uniform(-0.35, -0.20)  # 中等
            elif plat == 'Android':
                anomaly_impact = np.random.uniform(-0.15, -0.05)  # 轻微
            else:
                anomaly_impact = np.random.uniform(-0.05, 0.05)  # 基本正常
            
            row.append({
                'platform': plat,
                'version': ver,
                'n_users': n_users,
                'impact_pct': round(anomaly_impact * 100, 1)
            })
        matrix.extend(row)
    
    df_matrix = pd.DataFrame(matrix)
    
    # 透视表
    pivot = df_matrix.pivot_table(
        values='impact_pct',
        index='platform',
        columns='version',
        aggfunc='first'
    )
    
    print("交叉维度贡献度矩阵 (平台 × 版本)")
    print("数值 = 异常日活跃率变化(%)")
    print(pivot.to_string())
    
    return df_matrix

cross_matrix = cross_dimension_drill(users)

In [ ]:
# 步骤2.3：自动化异动检测——基于统计过程控制(SPC)

def auto_anomaly_detection(df, metric_col='dau', window=7, sigma_threshold=2.5):
    """
    基于滚动均值和标准差自动检测异动
    类似工业界的SPC控制图
    """
    values = df[metric_col].values
    
    anomalies = []
    for i in range(window, len(values)):
        window_vals = values[i-window:i]
        mean = np.mean(window_vals)
        std = np.std(window_vals)
        
        upper_bound = mean + sigma_threshold * std
        lower_bound = mean - sigma_threshold * std
        
        current = values[i]
        z_score = (current - mean) / std if std > 0 else 0
        
        if current > upper_bound or current < lower_bound:
            anomalies.append({
                'date': df['date'].iloc[i],
                'value': current,
                'expected': mean,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound,
                'z_score': z_score,
                'direction': 'up' if current > upper_bound else 'down'
            })
    
    df_anomalies = pd.DataFrame(anomalies)
    
    # 可视化
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df['date'], values, 'b-o', linewidth=1.5, markersize=4, label='DAU')
    
    # 绘制控制带
    means = [np.mean(values[max(0, i-window):i]) for i in range(window, len(values))]
    stds = [np.std(values[max(0, i-window):i]) for i in range(window, len(values))]
    upper = [m + sigma_threshold * s for m, s in zip(means, stds)]
    lower = [m - sigma_threshold * s for m, s in zip(means, stds)]
    
    x_range = df['date'].iloc[window:]
    ax.fill_between(x_range, lower, upper, alpha=0.2, color='gray', label=f'±{sigma_threshold}σ 控制带')
    
    # 标记异常点
    if len(df_anomalies) > 0:
        anomaly_dates = df_anomalies['date'].values
        anomaly_values = df_anomalies['value'].values
        ax.scatter(anomaly_dates, anomaly_values, c='r', s=100, zorder=5, label='检测到异常')
    
    ax.set_title('自动化异动检测 (滚动SPC)', fontsize=14)
    ax.set_xlabel('日期')
    ax.set_ylabel(metric_col)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return df_anomalies

# 检测异常
anomalies_detected = auto_anomaly_detection(df_anomaly)
if len(anomalies_detected) > 0:
    print(f"\n检测到 {len(anomalies_detected)} 个异动点:")
    for _, row in anomalies_detected.iterrows():
        print(f"  {row['date'].strftime('%Y-%m-%d')}: DAU={row['value']:,}, "
              f"Z-score={row['z_score']:.2f}, 方向={row['direction']}")
else:
    print("未检测到显著异动")

### 步骤3：挑战练习——全流程异动诊断模拟

3.1 从原始日志到诊断报告的完整流水线
3.2 多假设并行验证
3.3 生成自动化诊断报告

In [ ]:
# 步骤3.1：完整诊断流水线

class DAUAnomalyDiagnoser:
    """
    DAU异动诊断器——将完整诊断流程封装为可复用的类
    """
    def __init__(self, df_dau, df_users):
        self.df_dau = df_dau
        self.df_users = df_users
        self.diagnosis_steps = []
    
    def step1_confirm_data(self):
        """Step 1: 数据确认"""
        latest = self.df_dau.iloc[-1]
        prev = self.df_dau.iloc[-2]
        
        checks = {
            '数据完整性': latest['dau'] > 0,
            '波动合理性': abs(latest['dau'] - prev['dau']) / prev['dau'] < 0.5,
            '星期效应': not (latest['is_weekend'] and prev['is_weekend'] == False and abs(latest['dau'] - prev['dau']) / prev['dau'] < 0.1)
        }
        
        self.diagnosis_steps.append({'step': '数据确认', 'result': checks})
        
        print("[Step 1] 数据确认")
        for check, result in checks.items():
            status = '✓' if result else '⚠ 需关注'
            print(f"  {status} {check}")
        return all(checks.values())
    
    def step2_pds(self):
        """Step 2: PDS拆解"""
        result = pds_decomposition(self.df_dau)
        self.diagnosis_steps.append({'step': 'PDS拆解', 'result': result})
        return result
    
    def step3_drilldown(self, dimensions=['platform', 'app_version', 'channel']):
        """Step 3: 维度下钻"""
        drill_results = {}
        for dim in dimensions:
            drill_results[dim] = dimension_drill_down(self.df_users, None, dimension=dim)
        self.diagnosis_steps.append({'step': '维度下钻', 'result': drill_results})
        return drill_results
    
    def step4_hypothesis(self, pds_result, drill_results):
        """Step 4: 假设生成与排序"""
        main_cause = pds_result['main_cause']
        
        hypotheses = []
        if main_cause == '留存用户':
            hypotheses = [
                {'hypothesis': '登录服务异常(老用户token失效)', 'prior': 0.35, 'check': '查服务端登录成功率'},
                {'hypothesis': 'Push推送服务故障(未触发召回)', 'prior': 0.25, 'check': '查Push到达率和点击率'},
                {'hypothesis': 'CDN故障导致页面加载超时', 'prior': 0.20, 'check': '查页面加载时间P99'},
                {'hypothesis': '竞品大促活动吸引用户', 'prior': 0.12, 'check': '查应用商店排名变化'},
                {'hypothesis': '数据埋点故障(实际DAU未降)', 'prior': 0.08, 'check': '查客户端上报量和服务端日志量'}
            ]
        elif main_cause == '新增用户':
            hypotheses = [
                {'hypothesis': '主要投放渠道预算耗尽', 'prior': 0.30, 'check': '查各渠道消耗金额'},
                {'hypothesis': '应用商店排名突然下降', 'prior': 0.25, 'check': '查ASO排名'},
                {'hypothesis': '新版本审核被拒/下架', 'prior': 0.20, 'check': '查应用商店状态'},
                {'hypothesis': '竞品大规模投放抢占市场', 'prior': 0.15, 'check': '查竞品投放趋势'},
                {'hypothesis': '渠道回传数据延迟', 'prior': 0.10, 'check': '查渠道API回传状态'}
            ]
        else:
            hypotheses = [
                {'hypothesis': '回流活动到期/结束', 'prior': 0.35, 'check': '查最近活动日历'},
                {'hypothesis': 'Push召回策略变更', 'prior': 0.30, 'check': '查Push策略后台'},
                {'hypothesis': '季节性因素(节假日结束)', 'prior': 0.20, 'check': '查去年同期数据'},
                {'hypothesis': '竞品推出新功能吸引回流', 'prior': 0.15, 'check': '查竞品动态'}
            ]
        
        # 按先验概率排序
        hypotheses.sort(key=lambda x: x['prior'], reverse=True)
        
        self.diagnosis_steps.append({'step': '假设验证', 'result': hypotheses})
        
        print("\n[Step 4] 假设列表（按先验概率排序）")
        for i, h in enumerate(hypotheses):
            print(f"  H{i+1} [{h['prior']:.0%}]: {h['hypothesis']}")
            print(f"       验证方法: {h['check']}")
        
        return hypotheses
    
    def step5_conclusion(self, pds_result, top_hypothesis):
        """Step 5: 生成诊断结论"""
        print("\n" + "="*60)
        print("异动诊断结论")
        print("="*60)
        print(f"异动类型: DAU {pds_result['dau_change_pct']:+.1f}%")
        print(f"主要问题: {pds_result['main_cause']}")
        print(f"最可能原因: {top_hypothesis['hypothesis']}")
        print(f"置信度: {top_hypothesis['prior']:.0%}")
        print(f"建议行动: {top_hypothesis['check']}")
        print(f"\n如果该假设被证伪，下一步验证H2")
    
    def run(self):
        """执行完整诊断流程"""
        print("="*60)
        print("DAU异动自动诊断系统 v1.0")
        print("="*60)
        
        # Step 1
        data_ok = self.step1_confirm_data()
        if not data_ok:
            print("\n⚠ 数据质量存疑，请先确认数据源")
            return
        
        # Step 2
        print("\n[Step 2] PDS拆解")
        pds_result = self.step2_pds()
        
        # Step 3
        print("\n[Step 3] 维度下钻")
        drill_results = self.step3_drilldown()
        
        # Step 4
        hypotheses = self.step4_hypothesis(pds_result, drill_results)
        
        # Step 5
        self.step5_conclusion(pds_result, hypotheses[0])
        
        return self.diagnosis_steps

# 运行完整诊断
diagnoser = DAUAnomalyDiagnoser(df_anomaly, users)
steps = diagnoser.run()

In [ ]:
# 步骤3.2：多假设并行验证——贝叶斯更新

def bayesian_hypothesis_update(hypotheses, evidence):
    """
    用贝叶斯定理更新各假设的后验概率
    evidence: {'hypothesis_index': likelihood_ratio}
    likelihood_ratio > 1 表示证据支持该假设
    """
    priors = np.array([h['prior'] for h in hypotheses])
    
    # 默认无信息证据
    likelihoods = np.ones(len(hypotheses))
    for idx, lr in evidence.items():
        likelihoods[idx] = lr
    
    # 贝叶斯更新
    unnormalized = priors * likelihoods
    posteriors = unnormalized / unnormalized.sum()
    
    for i, (h, post) in enumerate(zip(hypotheses, posteriors)):
        h['posterior'] = post
    
    print("\n贝叶斯信念更新:")
    print(f"{'假设':<35}{'先验':>8}{'后验':>8}{'变化':>8}")
    print("-"*60)
    for i, h in enumerate(hypotheses):
        prior = h['prior']
        posterior = h.get('posterior', prior)
        delta = posterior - prior
        print(f"H{i+1}: {h['hypothesis']:<30} {prior:>7.0%} {posterior:>7.0%} {delta:>+7.0%}")
    
    return hypotheses

# 模拟：获取到证据"登录成功率跌至60%"
sample_hypotheses = [
    {'hypothesis': '登录服务异常', 'prior': 0.35},
    {'hypothesis': 'Push服务故障', 'prior': 0.25},
    {'hypothesis': 'CDN故障', 'prior': 0.20},
    {'hypothesis': '竞品活动', 'prior': 0.12},
    {'hypothesis': '埋点故障', 'prior': 0.08}
]

# 观察到的证据：登录成功率从99%跌到60%
evidence = {0: 8.0, 1: 0.5, 2: 0.3, 3: 0.1, 4: 0.05}
updated = bayesian_hypothesis_update(sample_hypotheses, evidence)
print(f"\n→ H1（登录服务异常）的后验概率从35%飙升到{updated[0]['posterior']:.0%}")
print("→ 可以99%确定问题是登录服务异常")

In [ ]:
# 步骤3.3：异动归因因子分解——精确量化各维度贡献

def factor_decomposition(before_values, after_values, factor_names):
    """
    对总变化量进行因子分解
    使用对数分解法(Logarithmic Decomposition)
    """
    total = sum(before_values)
    total_after = sum(after_values)
    total_change_pct = (total_after - total) / total
    
    contributions = []
    for i, name in enumerate(factor_names):
        before_share = before_values[i] / total if total > 0 else 0
        after_share = after_values[i] / total_after if total_after > 0 else 0
        
        # 对数贡献度
        if before_share > 0 and after_share > 0:
            log_contrib = (after_share - before_share) / (np.log(after_share) - np.log(before_share)) * np.log((1 + total_change_pct))
        else:
            log_contrib = (after_values[i] - before_values[i]) / total
        
        contributions.append({
            'factor': name,
            'before': before_values[i],
            'after': after_values[i],
            'change': after_values[i] - before_values[i],
            'contribution': log_contrib
        })
    
    df_contrib = pd.DataFrame(contributions)
    
    print("因子归因分解（对数法）")
    print(f"总变化率: {total_change_pct:+.2%}")
    print(f"\n{'因子':<15}{'变化量':>10}{'贡献度':>10}")
    print("-"*35)
    for _, row in df_contrib.iterrows():
        print(f"{row['factor']:<15}{row['change']:>+10,}{row['contribution']:>+10.2%}")
    
    return df_contrib

# 使用示例：按平台分解DAU变化
before_platform = [28000, 30000, 8000]  # Day7各平台DAU
after_platform = [27000, 12000, 3000]   # Day8各平台DAU
decomp = factor_decomposition(before_platform, after_platform, ['iOS', 'Android', 'Web'])

## 📊 区块4：业务场景应用

### 场景1：周一早上的DAU骤降

**背景**：周一早上10点，运营总监在群里@你："昨天DAU 102万，前天120万。少了15%。帮我看看怎么回事。"你只有1小时给出初步结论。

**诊断时间线**（真实还原）：

**10:00-10:05（数据确认）**：
- 检查数据延迟：昨天的数据已完全入库 ✓
- 检查口径：DAU用user_id计算，口径前后一致 ✓
- 检查埋点：客户端上报量没有异常下降 ✓
- 排除"假异动"，进入正式分析

**10:05-10:15（PDS拆解）**：
- 新增用户：基本稳定（±15%波动）
- 留存用户：从66k暴跌到42k（-36%）← 问题在这！贡献了89%的降幅
- 回流用户：基本稳定

**10:15-10:35（维度下钻）**：
- 按平台拆：iOS -4%，Android -60%，Web -62%
- 按版本拆：v3.3.1暴跌（这个版本昨天灰度了30%用户）
- 交叉确认：Android + v3.3.1中，活跃度从18%降到5%

**10:35-10:50（假设验证）**：
- 假设1：v3.3.1引入了登录bug → 查服务端日志 → 登录成功率从99.2%降到61%
- 确认：v3.3.1中token刷新逻辑有bug，老用户token过期后无法自动刷新
- 影响面：v3.3.1灰度了30%的Android用户 + 20%的Web用户

**10:50-11:00（写结论）**：
> DAU下降15%的根因是v3.3.1的token刷新bug，导致30%已有用户无法正常登录。建议：（1）立即回滚v3.3.1（2）对受影响用户发送补偿Push（3）建立发版前登录回归测试用例。

**启示**：1小时内完成诊断并给出可操作的建议，比花3天做"DAU下降全面分析报告"有价值100倍。

### 场景2：周四晚的"幽灵下降"

**背景**：每周四晚上DAU都会"神秘下降"5-8%，周五早上恢复。持续了3个月，没人知道为什么。

**分析过程**：
1. 先确认不是数据问题——ETL任务正常
2. PDS拆解发现：回流用户周四晚特别低
3. 按回流来源拆：Push回流的用户周四晚缺失
4. 查Push系统：周四晚上是Push系统的"静默维护窗口"，停止推送4小时
5. 所以周四晚来不了回流用户，DAU自然就低了

**结论**：这不是"幽灵"，是Push系统维护窗口的已知影响。建议在DAU监控中加入"Push系统维护日"的特殊标记，避免重复报警。

### 场景3：DAU下降但所有部门都说"不是我们"

**背景**：DAU连续3周下降，问了所有部门——技术说服务正常，运营说活动量没少，市场说投放预算没减，产品说没改功能。

**分析方法**：
1. 这种"所有人都说没问题但数据在降"的场景，最可能的原因是结构性变化
2. 做同期群分析：按注册月份分组，看每个同期群的活跃度趋势
3. 发现：最近3个月的新注册用户，第2周的留存率比之前低了40%
4. 问题找到了：渠道在3个月前换了一批素材，CTR涨了但用户质量暴跌
5. 因为"质量下降"是缓慢显现的（老用户还在，新用户慢慢替换老用户），所以需要3个月才能在宏观DAU上看到

**启示**：渠道投放的"质量债"可能会延迟3个月才体现在DAU上。不能只看CTR和获客成本，必须追踪新用户的生命周期留存。

### 场景4：节假日后的"不可能恢复"

**背景**：春节后DAU从峰值的150万跌到105万，跌幅-30%。产品VP说"这不可能，我们做了大量春节留存活动"。

**分析过程**：
1. 先拉出去年同期的数据——去年春节后DAU从100万跌到70万，跌幅也是-30%
2. 所以今年的跌幅比例和去年一样，说明产品没有变好也没有变差
3. 但是！今年的DAU绝对值（105万）比去年（70万）高了50%
4. 而且今年的节后恢复速度比去年快——去年用了15天恢复到90万，今年只用6天恢复到130万
5. 所以VP的感受是错的——产品确实在变好，只是他的内心基准是"春节峰值"而不是"去年同期"

**启示**：绝对值下降30%可能不是坏事——如果去年同期下降了30%，今年降了30%但绝对值比去年高50%。分析师的职责是提供正确的比较基准。

### 场景5：竞争对手上线新功能的连锁反应

**背景**：竞品6月1日上线了一个和你家产品非常相似的"免费版"，你的DAU从6月3日开始出现下降趋势。

**分析框架**：
1. **确定影响面**：按用户重合度分群——你和竞品都装的用户 vs 只装你家的用户
2. **确定影响深度**：竞品上线后，重合用户的活跃天数下降了多少
3. **确定用户画像**：流失到竞品的用户有什么特征（价格敏感？功能诉求？）
4. **制定对策**：
   - 针对价格敏感用户→推出差异化功能（免费版没有的高级功能）
   - 针对功能诉求用户→加速Roadmap中的核心功能
   - 监测竞品留存率→看竞品的"免费版"能否留住用户（如果能，威胁更大）

**启示**：竞品变化导致的DAU下降，分析重点不是"降了多少"，而是"哪些用户在流失"和"我们能做什么"。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出PDS拆解的三个人群及各自含义
- [ ] 能在1小时内完成一次DAU异动诊断（数据确认→PDS→下钻→假设→结论）
- [ ] 能区分"数据问题"和"业务问题"导致的指标异动
- [ ] 能用一个真实案例讲清楚异动诊断的完整流程

**答案提示**：
- PDS三人组：新增(N)、留存(R)、回流(C)
- 诊断流程五步走：数据→PDS→维度→假设→结论
- 数据延迟/口径变更/埋点故障属于"假异动"

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释贡献度分析的数学原理（为什么不能只看变化量）
- [ ] 能识别并规避辛普森悖论在异动分析中的陷阱
- [ ] 能设计多维度交叉下钻策略
- [ ] 能用贝叶斯更新来量化多假设的置信度变化
- [ ] 能区分骤降型、缓降型、周期型、结构型四种异动模式

**答案提示**：
- 贡献度 = 某人群变化量/总变化量，归一化后才可比
- 辛普森悖论：总量下降但每个子维度都在涨→检查人口结构变化
- 交叉下钻：找到"平台×版本"组合中受影响最严重的子群

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能设计自动化异动检测系统（SPC控制图/3-sigma规则/Prophet异常检测）
- [ ] 能在30分钟内给出80%置信度的诊断结论（而不是3小时后给95%）
- [ ] 能处理"所有维度拆完都没发现问题"的困境（结构性变化的滞后效应）
- [ ] 能建立异动诊断的SOP并培训新同事
- [ ] 能判断"异动是否需要立即响应"（区分日常波动和真正异常）

**答案提示**：
- 自动化检测用SPC控制图±2.5σ阈值
- 30分钟诊断要果断剪枝，只追贡献>50%的维度
- 维度全正常→结构性变化（同期群分析）
- SOP标准化可以减少每次重复思考的时间

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 第1周SQL窗口函数与JOIN | PDS拆解需要GROUP BY和窗口函数 |
| **← 前置依赖** | 第3周AB实验 | 判断变化是否统计显著 |
| **← 前置依赖** | 第6周用户增长与同期群 | 同期群视角定位结构性变化 |
| **← 前置依赖** | 第8周指标体系设计 | 有好的指标体系才有快速的异动定位 |
| **→ 后续用途** | W11综合项目 | 综合项目中包含完整的异动诊断任务 |
| **→ 后续用途** | 实际工作中的日报/周报 | 每天都需要用这套方法论看数据 |
| **业务连接** | 数据质量监控 | 异动诊断能力直接转化为监控规则 |
| **业务连接** | 应急响应流程 | 诊断结论直接触发oncall和回滚决策 |

### 横向连接：与其他分析方法的关联

**与指标体系设计的连接**：好的北极星指标+护栏指标设计，可以在异动发生时立刻给你提供context。比如你把DAU定义为北极星指标，那么PDS拆解和维度下钻就是自动化的护栏指标分析。

**与AB实验的连接**：异动诊断和AB实验本质上都是"归因问题"——AB实验是有对照组的归因（可控制），异动诊断是没有对照组的归因（只能靠拆解和假设验证）。两者的核心方法论是相通的。

**与用户增长的连接**：PDS拆解的三人群（新增/留存/回流）正好对应AARRR模型中的Acquisition、Retention、Revenue（回流用户的价值再捕获）。异动诊断本质上就是看AARRR的哪个环节出了问题。

**与数据分析软技能的连接**：异动诊断最能体现"在有限时间内给出最有信息量的判断"这一核心能力。这也是面试中最喜欢考察的能力——因为它综合考察了技术能力、业务理解和沟通能力。

## 📝 区块7：当日小结

> **异动诊断"三板斧"：先PDS拆解定位到"是哪块用户出了问题"，再维度下钻定位到"是哪种用户/哪个平台"，最后假设验证找到根因。80%的DAU异动能在2步内找到方向。记住：1小时给出80%置信度的方向 > 3天给出95%置信度的完整报告。真实工作中的分析不是"找到了完美的答案再汇报"，而是"在有限时间内给出最有信息量的判断"。**

**五条金句带回家**：
1. PDS拆解是异动诊断的第0步——先搞清楚是谁变了，再搞清楚为什么变
2. 30%的"异动"其实是数据问题
3. 维度下钻要按优先级有序进行，每次只追贡献最大的那个分支
4. 假设验证要找"证伪"数据，而不是"证实"数据
5. 分析的终点不是"找到了原因"，而是"提出了可验证的行动建议"

### 🔖 课后拓展

1. **练习**：下载你常用的App，拉取它的应用商店评分/评论趋势，如果某天评分突然下降0.5分，你如何诊断？

2. **思考题**：如果发现DAU下降了15%但这个变化在统计学上不显著（p=0.07），你应该怎样向运营总监汇报？

3. **预习**：明天的案例——GMV涨了30%但利润降了10%，涉及多维指标拆解和ROI分析

4. **面试准备**：背诵以下面试场景的回答思路：
   - "DAU跌了20%，给你1小时，你怎么排查？"
   - "如何区分技术故障和业务变化导致的DAU下降？"
   - "如果所有维度拆完后都没发现问题，下一步做什么？"

---

*03analyst_thinking · Week 10 Day 1 · 2026-05-21*